In [1]:
!pip install seaborn


  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached numpy-1.24.4-cp38-cp38-win_amd64.whl.metadata (5.6 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached numpy-1.24.4-cp38-cp38-win_amd64.whl (14.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.19.5
    Uninstalling numpy-1.19.5:
      Successfully uninstalled numpy-1.19.5


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-gpu 2.5.0 requires numpy~=1.19.2, but you have numpy 1.24.4 which is incompatible.
tensorflow-gpu 2.5.0 requires typing-extensions~=3.7.4, but you have typing-extensions 4.12.2 which is incompatible.


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, jaccard_score,
    cohen_kappa_score, roc_auc_score, roc_curve, confusion_matrix
)
import numpy as np

def evaluate_model_binary(pred_val, y_test, test_indices=None, threshold=0.60, df=None):

    # Predict
    style_pred_val_binary = (pred_val > threshold).astype(int)

    # Identify False Positives
    false_positive_indices = []
    if test_indices is not None:
        false_positive_indices = [
            test_indices[i] for i, (true, pred) in enumerate(zip(y_test, style_pred_val_binary))
            if true == 0 and pred == 1
        ]

    if df is not None and false_positive_indices:
        for idx in false_positive_indices:
            attributes_img1 = json.loads(df.loc[idx, 'attributes_img1'])
            attributes_img2 = json.loads(df.loc[idx, 'attributes_img2'])
            print(f"False Positive Pair {idx}:")
            print(f"Attributes Image 1: {attributes_img1}")
            print(f"Attributes Image 2: {attributes_img2}")

    # Metrics
    results = {
        'accuracy': accuracy_score(y_test, style_pred_val_binary),
        'precision': precision_score(y_test, style_pred_val_binary),
        'recall': recall_score(y_test, style_pred_val_binary),
        'f1_score': f1_score(y_test, style_pred_val_binary),
        'jaccard': jaccard_score(y_test, style_pred_val_binary),
        'kappa': cohen_kappa_score(y_test, style_pred_val_binary),
        'roc_auc': roc_auc_score(y_test, style_pred_val),
    }

    # Print metrics
    print(f"Accuracy: {results['accuracy']:.2f}")
    print(f"Precision: {results['precision']:.2f}")
    print(f"Recall: {results['recall']:.2f}")
    print(f"F1 Score: {results['f1_score']:.2f}")
    print(f"Jaccard Similarity: {results['jaccard']:.2f}")
    print(f"Cohen's Kappa: {results['kappa']:.2f}")
    print(f"ROC AUC: {results['roc_auc']:.2f}")

    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, pred_val)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {results["roc_auc"]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.title("ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

    # Confusion matrix
    cm = confusion_matrix(y_test, style_pred_val_binary)
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Predicted: 0", "Predicted: 1"],
                yticklabels=["Actual: 0", "Actual: 1"])
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return results

########EXAMPLE USAGE
# results = evaluate_model_binary(
#     style_pred_val,
#     y_test=y_style_test,
#     test_indices=test_indices,  # optional
#     threshold=0.60,
#     df=df  # optional, if you want to print false positive attribute analysis
# )
